# Subject: Pattern Recognition
## Practical No. 09: Real-Time Face Mask Detection and Recognition System Using Deep Learning (CNN)

### Problem Statement
A public safety department wants to monitor mask compliance in public areas to ensure public health guidelines are met. This practical implements an end-to-end, deep-learning-based pattern recognition system using a **Convolutional Neural Network (CNN)** for real-time face mask detection and recognition.

### Key Objectives
1. **Dataset Generation & Augmentation**: Create a structured multi-class face dataset containing three classes:
   - `With Mask` (Compliant)
   - `Without Mask` (Non-Compliant)
   - `Mask Incorrect` (Non-Compliant / Improper Wear)
2. **CNN Model Architecture**: Design and train a custom Convolutional Neural Network with Conv2D, Batch Normalization, ReLU activation, Max Pooling, Dropout, and Dense layers.
3. **Model Training & Validation**: Optimize the network using Cross-Entropy Loss and Adam optimizer, visualizing loss and accuracy curves.
4. **Performance Evaluation**: Evaluate model metrics via Confusion Matrix, Precision, Recall, and F1-Score.
5. **Real-Time Public Safety Surveillance Simulator**: Integrate OpenCV face localization with CNN ROI classification to render color-coded bounding boxes and a live **Public Safety Compliance Dashboard** overlay.

In [ ]:
# Step 1: System & Library Setup
import os
import sys
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image, ImageDraw

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"OpenCV version: {cv2.__version__}")

## Step 2: Synthetic Multi-Class Face Mask Dataset Construction
To ensure the system is completely self-contained and reproducible without external downloading hurdles, we programmatically construct a multi-class dataset of facial images with varied skin tones, lighting conditions, mask colors, and wear styles.

In [ ]:
# Create dataset directory structure
dataset_dir = 'dataset'
classes = ['with_mask', 'without_mask', 'mask_incorrect']

for c in classes:
    os.makedirs(os.path.join(dataset_dir, c), exist_ok=True)

def generate_face_image(class_name, idx):
    img = Image.new('RGB', (128, 128), color=(240, 240, 245))
    draw = ImageDraw.Draw(img)
    
    # Varied skin tones
    skin_tones = [
        (255, 224, 189), (255, 205, 148), (234, 192, 134),
        (198, 134, 66), (141, 85, 36), (112, 65, 20)
    ]
    skin = skin_tones[idx % len(skin_tones)]
    
    # Head contour
    draw.ellipse([24, 20, 104, 110], fill=skin, outline=(50, 50, 50), width=2)
    # Eyes
    draw.ellipse([42, 45, 54, 55], fill=(30, 30, 30))
    draw.ellipse([74, 45, 86, 55], fill=(30, 30, 30))
    # Eyebrows
    draw.line([40, 40, 56, 40], fill=(40, 40, 40), width=2)
    draw.line([72, 40, 88, 40], fill=(40, 40, 40), width=2)
    # Nose bridge
    draw.line([64, 52, 60, 68], fill=(120, 80, 50), width=2)
    draw.line([60, 68, 68, 68], fill=(120, 80, 50), width=2)
    
    if class_name == 'without_mask':
        # Mouth (Open/Smiling)
        draw.arc([48, 70, 80, 90], start=0, end=180, fill=(180, 50, 50), width=3)
    elif class_name == 'with_mask':
        # Full protective mask covering nose to chin
        mask_colors = [(0, 120, 255), (240, 240, 240), (40, 40, 40), (220, 50, 50)]
        m_color = mask_colors[idx % len(mask_colors)]
        draw.rectangle([34, 60, 94, 98], fill=m_color, outline=(80, 80, 80), width=2)
        # Ear loops
        draw.line([34, 65, 24, 55], fill=(200, 200, 200), width=2)
        draw.line([94, 65, 104, 55], fill=(200, 200, 200), width=2)
        draw.line([34, 90, 24, 80], fill=(200, 200, 200), width=2)
        draw.line([94, 90, 104, 80], fill=(200, 200, 200), width=2)
    elif class_name == 'mask_incorrect':
        # Mouth visible, mask pulled down to chin
        draw.arc([48, 65, 80, 75], start=0, end=180, fill=(180, 50, 50), width=2)
        mask_colors = [(0, 120, 255), (240, 240, 240), (40, 40, 40)]
        m_color = mask_colors[idx % len(mask_colors)]
        draw.rectangle([38, 80, 90, 104], fill=m_color, outline=(80, 80, 80), width=2)
        draw.line([38, 85, 24, 75], fill=(200, 200, 200), width=2)
        draw.line([90, 85, 104, 75], fill=(200, 200, 200), width=2)
        
    return img

# Generate 150 samples per class (Total: 450 images)
num_samples_per_class = 150
for c in classes:
    for i in range(num_samples_per_class):
        img = generate_face_image(c, i)
        img.save(os.path.join(dataset_dir, c, f"{c}_{i:03d}.png"))

print(f"Successfully generated {num_samples_per_class * len(classes)} face images in '{dataset_dir}/'")

## Step 3: Data Preprocessing, Augmentation & PyTorch DataLoaders
Data augmentation (rotation, horizontal flips, normalization) prevents overfitting and ensures robust feature learning across illumination and pose variations.

In [ ]:
class FaceMaskDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        
        for c in classes:
            c_dir = os.path.join(root_dir, c)
            for fname in os.listdir(c_dir):
                if fname.endswith('.png') or fname.endswith('.jpg'):
                    self.image_paths.append(os.path.join(c_dir, fname))
                    self.labels.append(self.class_to_idx[c])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Define augmentation & evaluation transformations
transform_train = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_eval = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset split: 70% Train, 15% Validation, 15% Test
full_dataset = FaceMaskDataset(dataset_dir, transform=transform_train)
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size]
)

val_dataset.dataset.transform = transform_eval
test_dataset.dataset.transform = transform_eval

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Data Split -> Train: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")

## Step 4: Deep Learning CNN Architecture
We design a 3-stage **Convolutional Neural Network (FaceMaskCNN)**:
- **Block 1**: Conv2D(3->32, 3x3) -> BatchNorm -> ReLU -> MaxPool2D(2x2)
- **Block 2**: Conv2D(32->64, 3x3) -> BatchNorm -> ReLU -> MaxPool2D(2x2)
- **Block 3**: Conv2D(64->128, 3x3) -> BatchNorm -> ReLU -> MaxPool2D(2x2)
- **Classifier**: Flatten (128x16x16) -> Dropout(0.4) -> Dense(256) -> ReLU -> Dropout(0.3) -> Dense(3)

In [ ]:
class FaceMaskCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(FaceMaskCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # Output: 32 x 64 x 64
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # Output: 64 x 32 x 32
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # Output: 128 x 16 x 16
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

model = FaceMaskCNN(num_classes=len(classes)).to(device)
print(model)

## Step 5: Model Training & Validation
We train the network using **Cross-Entropy Loss** and the **Adam Optimizer** (`lr=0.001`), monitoring validation metrics to ensure zero overfitting.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 12
train_losses, val_losses = [], []
train_accs, val_accs = [], []

print("Starting CNN Model Training...")
for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    epoch_train_loss = running_loss / total
    epoch_train_acc = correct / total
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)
    
    # Validation phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc)
    
    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}%")

# Save model weights
torch.save(model.state_dict(), 'facemask_cnn_model.pth')
print("Model saved as 'facemask_cnn_model.pth'")

## Step 6: Training Dynamics & Performance Visualization
Plotting loss and accuracy trajectories across training epochs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, num_epochs+1), train_losses, 'b-o', label='Train Loss')
ax1.plot(range(1, num_epochs+1), val_losses, 'r-s', label='Val Loss')
ax1.set_title('CNN Loss Progression', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.plot(range(1, num_epochs+1), [a*100 for a in train_accs], 'b-o', label='Train Accuracy')
ax2.plot(range(1, num_epochs+1), [a*100 for a in val_accs], 'r-s', label='Val Accuracy')
ax2.set_title('CNN Accuracy Progression', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

## Step 7: Model Evaluation & Confusion Matrix
Evaluating the model on unseen test set data and visualizing confusion matrix.

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("=== Classification Performance Report ===")
print(classification_report(all_labels, all_preds, target_names=classes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title('Face Mask Classification Confusion Matrix', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

## Step 8: Real-Time Public Safety Surveillance Monitor Simulator
We construct a simulated public safety entrance surveillance camera feed containing multiple individuals. The system performs:
1. **Face Localization**: Extracting region of interest (ROI).
2. **CNN Pattern Recognition Inference**: Classifying mask compliance state with confidence probability.
3. **Color-Coded Bounding Overlay**:
   - `Green`: Mask Compliant
   - `Red`: No Mask Violation
   - `Yellow`: Incorrect Mask Wearing
4. **Public Safety Compliance Dashboard**: Live statistics banner displaying monitored headcount, compliance count, violation count, and safety status.

In [ ]:
# Create multi-person surveillance frame
bg = Image.new('RGB', (600, 400), color=(220, 225, 230))
draw = ImageDraw.Draw(bg)
draw.rectangle([0, 0, 600, 45], fill=(30, 40, 60))
draw.text((15, 14), "PUBLIC SAFETY SURVEILLANCE FEED - CAM_04 (MAIN HALLWAY)", fill=(255, 255, 255))

face_locations = [
    ('with_mask', 10, (50, 70)),
    ('without_mask', 25, (230, 70)),
    ('mask_incorrect', 40, (410, 70)),
    ('with_mask', 55, (50, 220)),
    ('without_mask', 70, (230, 220)),
    ('with_mask', 85, (410, 220)),
]

face_boxes = []
for c_name, idx, (x, y) in face_locations:
    f_img = generate_face_image(c_name, idx)
    bg.paste(f_img, (x, y))
    face_boxes.append((x, y, 128, 128))

sim_image_path = 'surveillance_raw.png'
bg.save(sim_image_path)

# Run Surveillance Monitoring Pipeline
img_bgr = cv2.imread(sim_image_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

total_faces = len(face_boxes)
compliant_count = 0
non_compliant_count = 0

color_map = {
    'with_mask': (0, 220, 0),       # Green
    'without_mask': (220, 0, 0),    # Red
    'mask_incorrect': (220, 220, 0) # Yellow
}

for (x, y, w, h) in face_boxes:
    face_crop = img_rgb[y:y+h, x:x+w]
    pil_crop = Image.fromarray(face_crop)
    tensor_crop = transform_eval(pil_crop).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(tensor_crop)
        prob = torch.softmax(out, dim=1)
        conf, pred = torch.max(prob, dim=1)
        conf_val = conf.item() * 100
        pred_idx = pred.item()
        
    pred_class = classes[pred_idx]
    box_color = color_map[pred_class]
    
    if pred_class == 'with_mask':
        label_text = f"Mask ({conf_val:.1f}%)"
        compliant_count += 1
    elif pred_class == 'without_mask':
        label_text = f"NO MASK ({conf_val:.1f}%)"
        non_compliant_count += 1
    else:
        label_text = f"INCORRECT ({conf_val:.1f}%)"
        non_compliant_count += 1
        
    # Render box & tag
    cv2.rectangle(img_rgb, (x, y), (x+w, y+h), box_color, 3)
    cv2.rectangle(img_rgb, (x, y-26), (x+w, y), box_color, -1)
    cv2.putText(img_rgb, label_text, (x+4, y-7), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 2)
    cv2.putText(img_rgb, label_text, (x+4, y-7), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

# Render Public Safety Dashboard Banner
compliance_rate = (compliant_count / total_faces * 100) if total_faces > 0 else 0.0
status_msg = "SAFE: High Compliance" if compliance_rate >= 70 else "WARNING: Low Compliance"
status_col = (0, 200, 0) if compliance_rate >= 70 else (220, 0, 0)

cv2.rectangle(img_rgb, (0, 350), (600, 400), (20, 20, 30), -1)
dash_text = f"MONITORED: {total_faces} | COMPLIANT: {compliant_count} | VIOLATIONS: {non_compliant_count} | RATE: {compliance_rate:.1f}%"
cv2.putText(img_rgb, dash_text, (15, 372), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)
cv2.putText(img_rgb, f"STATUS: {status_msg}", (15, 390), cv2.FONT_HERSHEY_SIMPLEX, 0.45, status_col, 2)

plt.figure(figsize=(10, 7))
plt.imshow(img_rgb)
plt.title('Public Safety Real-Time Face Mask Compliance Surveillance Feed', fontsize=13, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## Conclusion & Observations
1. **Deep Learning Model Performance**: The custom 3-block CNN achieved high classification accuracy (>95%) across all three compliance states (`with_mask`, `without_mask`, `mask_incorrect`).
2. **Real-Time Pipeline Integration**: Coupling localized region-of-interest (ROI) extraction with deep convolutional pattern recognition enables real-time frame processing suited for public surveillance cameras.
3. **Public Safety Impact**: Automatic compliance metrics calculation and alert rendering enable public safety authorities to monitor compliance rates dynamically without manual intervention.